In [ ]:
import math

def entropy(labels):
    """计算标签的熵（混乱程度）"""
    from collections import Counter
    counts = Counter(labels)
    total = len(labels)
    
    entropy_val = 0.0
    for count in counts.values():
        probability = count / total
        if probability > 0:
            entropy_val -= probability * math.log2(probability)
    return entropy_val

# 示例：计算原始数据的熵
labels = [1, 1, 0, 0, 1]  # 1:喜欢科幻, 0:不喜欢
print(f"原始熵: {entropy(labels):.3f}")


In [ ]:
def gini_impurity(labels):
    """计算基尼不纯度"""
    from collections import Counter
    counts = Counter(labels)
    total = len(labels)
    
    gini = 1.0
    for count in counts.values():
        probability = count / total
        gini -= probability ** 2
    return gini

# 示例
print(f"基尼不纯度: {gini_impurity(labels):.3f}")


In [ ]:
class SimpleDecisionTree:
    def __init__(self, max_depth=3):
        self.max_depth = max_depth
        self.tree = None
    
    def fit(self, X, y, features):
        """训练决策树"""
        self.tree = self._build_tree(X, y, features, depth=0)
    
    def _build_tree(self, X, y, features, depth):
        """递归构建树"""
        # 终止条件
        if depth >= self.max_depth or len(set(y)) == 1 or not features:
            # 返回叶节点：多数类别
            from collections import Counter
            majority_class = Counter(y).most_common(1)[0][0]
            return {'type': 'leaf', 'class': majority_class}
        
        # 选择最佳分割特征（简化版：随机选择）
        import random
        best_feature = random.choice(features)
        
        # 分割数据
        left_indices = [i for i, val in enumerate(X) if val[best_feature] == 1]
        right_indices = [i for i, val in enumerate(X) if val[best_feature] == 0]
        
        # 递归构建子树
        remaining_features = [f for f in features if f != best_feature]
        
        left_subtree = self._build_tree(
            [X[i] for i in left_indices],
            [y[i] for i in left_indices],
            remaining_features,
            depth + 1
        )
        
        right_subtree = self._build_tree(
            [X[i] for i in right_indices],
            [y[i] for i in right_indices],
            remaining_features,
            depth + 1
        )
        
        return {
            'type': 'node',
            'feature': best_feature,
            'left': left_subtree,
            'right': right_subtree
        }
    
    def predict(self, x):
        """预测单个样本"""
        node = self.tree
        while node['type'] == 'node':
            if x[node['feature']] == 1:
                node = node['left']
            else:
                node = node['right']
        return node['class']
    
    def display_tree(self, node=None, indent=""):
        """可视化决策树"""
        if node is None:
            node = self.tree
        
        if node['type'] == 'leaf':
            print(f"{indent}预测: {node['class']}")
        else:
            print(f"{indent}特征{node['feature']} = 1?")
            print(f"{indent}├── 是 → ", end="")
            self.display_tree(node['left'], indent + "│   ")
            print(f"{indent}└── 否 → ", end="")
            self.display_tree(node['right'], indent + "    ")
